<a href="https://colab.research.google.com/github/cbonnin88/EcoTrack/blob/main/EcoTrack_data_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install Faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 25.1 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import random
from faker import Faker
from datetime import timedelta

In [ ]:
fake = Faker()
Faker.seed(42)
random.seed(42)

In [ ]:
NUM_USERS = 25000
print('Starting EcoTrack Phase 0: Enhancing Data Generation....')

Starting EcoTrack Phase 0: Enhancing Data Generation....


# **Generate Users**

In [ ]:
print(f"Generating {NUM_USERS} Users with demographics and new tiers...")
countries = ['France', 'Germany', 'Netherlands', 'Belgium', 'Spain', 'Ireland']
tiers = ['Free', 'Basic', 'Premium']
tier_weights = [0.70, 0.20, 0.10]

Generating 25000 Users with demographics and new tiers...


In [ ]:
genders = ['Female', 'Male', 'Non-binary']
gender_weights = [0.49, 0.48, 0.03]

In [ ]:
users_data = []
for _ in range(NUM_USERS):
    user_id = fake.uuid4()
    # Account creation timestamp within the last 365 days
    created_at = fake.date_time_between(start_date='-1y', end_date='now')
    country = random.choice(countries)
    tier = random.choices(tiers, weights=tier_weights)[0]
    age = random.randint(18, 70)
    gender = random.choices(genders, weights=gender_weights)[0]

    users_data.append([user_id, created_at, country, tier, age, gender])

df_users = pd.DataFrame(users_data, columns=[
    'user_id', 'created_at', 'country', 'subscription_tier', 'age', 'gender'
])
df_users.to_csv('users.csv', index=False)

# **Generate Sessions**

In [ ]:
print("Generating App Sessions (Scaling realistic engagement)...")
sessions_data = []

for _, user in df_users.iterrows():
    # Realistic distribution: some users drop off immediately, power users engage heavily
    # Premium and Basic users generally exhibit higher retention/session frequency
    if user['subscription_tier'] == 'premium':
        num_sessions = random.choices([random.randint(1, 5), random.randint(6, 25), random.randint(26, 60)], weights=[0.1, 0.6, 0.3])[0]
    elif user['subscription_tier'] == 'basic':
        num_sessions = random.choices([random.randint(1, 5), random.randint(6, 20), random.randint(21, 40)], weights=[0.2, 0.6, 0.2])[0]
    else:
        num_sessions = random.choices([random.randint(1, 3), random.randint(4, 10), random.randint(11, 20)], weights=[0.5, 0.4, 0.1])[0]

    for _ in range(num_sessions):
        session_id = fake.uuid4()
        session_start = fake.date_time_between(start_date=user['created_at'], end_date='now')
        # Typical mobile app sessions last between 2 and 20 minutes
        session_end = session_start + timedelta(minutes=random.randint(2, 20))

        sessions_data.append([session_id, user['user_id'], session_start, session_end])

df_sessions = pd.DataFrame(sessions_data, columns=['session_id', 'user_id', 'session_start', 'session_end'])
df_sessions.to_csv('sessions.csv', index=False)

Generating App Sessions (Scaling realistic engagement)...


# **Generate Events**

In [ ]:
print("Generating Expanded Event logs...")
# Expanded taxonomy with realistic product tracking milestones
event_types = [
    'open_app', 'view_dashboard', 'log_habit', 'join_challenge',
    'read_article', 'share_milestone', 'view_upgrade_screen', 'purchase'
]
# Distribution weights favoring top-of-funnel interactions over conversion actions
event_weights = [0.35, 0.25, 0.20, 0.08, 0.06, 0.03, 0.02, 0.01]

events_data = []
for _, session in df_sessions.iterrows():
    # Scaling event frequency dynamically based on how long the user stayed in the session
    duration_minutes = (session['session_end'] - session['session_start']).seconds // 60
    num_events = max(1, min(12, int(duration_minutes * random.uniform(0.5, 1.5))))

    for _ in range(num_events):
        event_id = fake.uuid4()
        event_timestamp = fake.date_time_between(start_date=session['session_start'], end_date=session['session_end'])
        event_type = random.choices(event_types, weights=event_weights)[0]

        events_data.append([event_id, session['user_id'], session['session_id'], event_type, event_timestamp])

df_events = pd.DataFrame(events_data, columns=['event_id', 'user_id', 'session_id', 'event_type', 'event_timestamp'])
df_events.to_csv('events.csv', index=False)

Generating Expanded Event logs...


# **Generate Transactions**

In [ ]:
print("Generating Transactions mapped to purchases and tier plans...")
transactions_data = []

# Pricing tiers mapping
tier_prices = {'basic': 3.99, 'premium': 8.99}
item_types = ['carbon_offset', 'eco_merch']
merch_prices = [9.50, 19.99, 35.00, 49.99]

# 4a. Capture transaction events driven by user in-app actions ('purchase' events)
purchase_events = df_events[df_events['event_type'] == 'purchase']
for _, event in purchase_events.iterrows():
    transaction_id = fake.uuid4()
    item = random.choice(item_types)
    amount = random.choice(merch_prices) if item == 'eco_merch' else round(random.uniform(2.00, 25.00), 2)

    transactions_data.append([transaction_id, event['user_id'], event['event_timestamp'], amount, item])

# 4b. Ensure paid subscribers have a corresponding subscription transaction log
paid_users = df_users[df_users['subscription_tier'].isin(['basic', 'premium'])]
for _, user in paid_users.iterrows():
    transaction_id = fake.uuid4()
    # Subscription purchase transaction matches their initial sign-up time
    amount = tier_prices[user['subscription_tier']]
    item = f"subscription_{user['subscription_tier']}"

    transactions_data.append([transaction_id, user['user_id'], user['created_at'], amount, item])

df_transactions = pd.DataFrame(transactions_data, columns=['transaction_id', 'user_id', 'transaction_date', 'amount', 'item_type'])
df_transactions.to_csv('transactions.csv', index=False)

Generating Transactions mapped to purchases and tier plans...


In [ ]:
print(f"✅ Success! Created massive datasets in your working directory:")
print(f" -> users.csv ({len(df_users)} rows)")
print(f" -> sessions.csv ({len(df_sessions)} rows)")
print(f" -> events.csv ({len(df_events)} rows)")
print(f" -> transactions.csv ({len(df_transactions)} rows)")

✅ Success! Created massive datasets in your working directory:
 -> users.csv (25000 rows)
 -> sessions.csv (133365 rows)
 -> events.csv (1122181 rows)
 -> transactions.csv (11249 rows)
